In [ ]:
import os
from pathlib import Path
import pandas as pd

### Set paths:

In [ ]:

project_dir = Path(os.getcwd())

root_path = (project_dir / '..' / 'data_folders').resolve()

print(os.listdir(root_path))

swisstopo_path = root_path / 'swiss_topo_data_orig'

vg_path = root_path/'visual_genome_proc_data'  # Replace with your directory containing images

combined_data_path = root_path / 'combined_data'


In [ ]:

vg_labels_path = vg_path / 'labels.csv'
#swisstopo_labels_path = swisstopo_path / 'patches/labels.csv'
swisstopo_labels_path = swisstopo_path / 'patches/labels.csv'


# Load both CSVs
vg_meta = pd.read_csv(vg_labels_path)
swiss_meta = pd.read_csv(swisstopo_labels_path)

# Prepare Visual Genome: keep only image_id and file_paths, add is_map=0
vg_meta = vg_meta[['image_id', 'file_paths']].copy()
vg_meta['is_map'] = 0
vg_meta['file_source'] = 'visual_genome'

# Prepare swisstopo: rename label to is_map
swiss_meta = swiss_meta[['image_id', 'file_paths', 'label']].rename(columns={'label': 'is_map'})
swiss_meta['file_source'] = 'swiss_topo'

# Concatenate and save
combined_labels_path = combined_data_path / 'labels.csv'
combined_meta = pd.concat([vg_meta, swiss_meta], ignore_index=True)
combined_meta.to_csv(combined_labels_path, index=False)

print(f"Combined dataset: {len(combined_meta)} images ({(combined_meta.is_map == 0).sum()} photos, {(combined_meta.is_map == 1).sum()} maps)")

In [ ]:
label_data_path = combined_data_path / 'labels.csv'
label_data_path

In [ ]:
label_data_path

In [ ]:
import re
import os
import pandas as pd

In [ ]:
label_data = pd.read_csv(label_data_path)

In [ ]:
file_paths = list(label_data.file_paths)
selection_bools = []
for file_path in file_paths:
    selection_bool = ('swiss_topo' in file_path)
    selection_bools.append(selection_bool)
df = label_data[selection_bools]
df.shape

In [ ]:

# Extract trailing numbers from image_id and filename
# id_numbers = df['image_id'].str.extract(r'(\d+)$')[0].astype(int)
id_numbers = df['image_id']
filename_numbers = df['file_paths'].apply(lambda x: re.search(r'(\d+)\.jpg$', x).group(1)).astype(int)

# Check uniqueness
print("image_id numbers unique:", id_numbers.nunique() == len(df))
print("filename numbers unique:", filename_numbers.nunique() == len(df))

# Check they match
print("image_id and filename numbers match:", (id_numbers == filename_numbers).all())



In [ ]:
label_data.head()

In [ ]:
label_data.tail()

In [ ]:

missing = label_data[~label_data['file_paths'].apply(os.path.exists)]
print(f"{len(missing)} missing files out of {len(label_data)}")
print(missing['file_paths'].head(10))

In [ ]:
label_data.head()

In [ ]:
file_source_1 = 'visual_genome' 
file_source_2 = 'swiss_topo'

In [ ]:
filepaths = list(label_data.file_paths)
file_source_check = []
for filepath in filepaths:
    if file_source_1 in filepath:
        file_source_check.append(file_source_1)
    elif file_source_2 in filepath:
        file_source_check.append(file_source_2)

print(len(file_source_check))
print(len(filepaths))

In [ ]:
file_source = list(label_data.file_source)

In [ ]:
file_source == file_source_check